In [201]:
import pandas as pd
import numpy as np
import json
import sys

sys.path.append('/home/mdafifal.mamun/research/S3M')

In [202]:
prediction_path = "/home/mdafifal.mamun/research/S3M/saved_predictions/eclipse_dedupt_predictions_1771135140.5734932.json"

In [203]:
with open(prediction_path, 'r') as f:
    predictions = json.load(f)

In [204]:
dedupt_processed_results = []

for _, result in predictions.items():
    bug_id = result["bug_id"]
    actual_duplicate_id = result["actual_duplicate_id"]
    preds = [int(pred) for pred in result["predictions"].keys()]

    dedupt_processed_results.append((bug_id, actual_duplicate_id, preds))
    

In [206]:
rows = []

for dedupt_result in dedupt_processed_results:
    bug_id, actual_duplicate_id, dedupt_preds = dedupt_result
    
    rows.append({
        "bug_id": bug_id,
        "actual_duplicate_id": actual_duplicate_id,
        "dedupt_preds": dedupt_preds[:10],        
    })
result_df = pd.DataFrame(rows)

def reciprocal_rank(preds, actual):
    try:
        rank = preds.index(actual) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0


result_df["rr_dedupt"] = result_df.apply(
    lambda x: reciprocal_rank(x["dedupt_preds"], x["actual_duplicate_id"]),
    axis= 1
)

In [273]:
analytical_samples = result_df.sample(n=50, random_state=500).head(20)
analytical_samples["rr_dedupt"].mean()

0.8125

In [274]:
analytical_samples

,bug_id,actual_duplicate_id,dedupt_preds,rr_dedupt
299,444552,443906,"[443906, 433552, 417764, 223381, 281455, 43933...",1.00
504,446468,445457,"[445457, 445983, 444102, 445116, 445782, 44400...",1.00
1915,461384,441721,"[461119, 461018, 445579, 441721, 453149, 45280...",0.25
695,447164,394512,"[394512, 439159, 442783, 358491, 441692, 43406...",1.00
1979,464750,441721,"[441721, 461119, 462048, 461018, 445579, 45314...",1.00
1813,454187,448995,"[448995, 446782, 447383, 446029, 449408, 44693...",1.00
343,445256,443896,"[407376, 425420, 416267, 424727, 426778, 44325...",0.00
1946,462837,462208,"[456720, 444767, 441699, 462208, 455891, 44823...",0.25
545,446687,438945,"[426576, 438945, 428524, 433478, 424195, 42993...",0.50
1821,454712,442783,"[442783, 451648, 394512, 454109, 446724, 44169...",1.00


In [279]:
analytical_samples["rr_dedupt"].value_counts()

rr_dedupt
1.00    15
0.25     3
0.00     1
0.50     1
Name: count, dtype: int64

In [275]:
full_dataset = "/home/mdafifal.mamun/research/S3M/dataset/EMSE_data/eclipse_2018/eclipse_stacktraces.json"

with open(full_dataset, 'r') as f:
    full_data = json.load(f)

In [276]:
full_data_df = pd.DataFrame(full_data)

In [277]:
num_frames = 10
def format_stack(stack, from_top=False):
    # Remove duplicate frames
    exception = stack.get("exception", "UnknownException")
    message = stack.get("message", "")
    frames = stack.get("frames", [])
    frames = [f"{f['function']} at {f['file']}:{f['fileline']}" for f in frames]            

    frames = "\n".join([f"{i+1}: {frame}" for i, frame in enumerate(frames)][:num_frames])
    return f"{exception}: {message}\n{frames}"


def get_bug_repot(bug_id):
    bug_report = full_data_df[full_data_df["bug_id"] == bug_id]
    br = f"""
    Bug ID: {bug_report.iloc[0]['bug_id']}
    Title: {bug_report.iloc[0]['short_desc']}

{format_stack(bug_report.iloc[0]['stacktrace'][0])}
"""
    return br

In [278]:
def check_predictions(analytical_sample):
    bug_id = analytical_sample["bug_id"]
    preds = analytical_sample["dedupt_preds"]
    print(f"Actual Bug ID: {bug_id}")
    print(get_bug_repot(bug_id))
    print("*" * 50)
    print("Predicted Duplicates:")
    for pred in preds[:5]:
        print(get_bug_repot(pred))
        print("-" * 50)
    print("=" * 50)


for _, sample in analytical_samples.head(10).iterrows():
    print(f"Analyzing Bug ID: {sample['bug_id']} with Actual Duplicate ID: {int(sample['actual_duplicate_id'])}")
    check_predictions(sample)

Analyzing Bug ID: 444552 with Actual Duplicate ID: 443906
Actual Bug ID: 444552

    Bug ID: 444552
    Title: Internal Error (err_grp: 753aa95b)

org.eclipse.core.internal.resources.ResourceException: Resource '/com.codetrails.ctrlflow.codesearch.rcp/src-gen/com/codetrails/ctrlflow/codesearch/rcp/model/ArgStatmentSnippet.java' does not exist.
1: org.eclipse.core.internal.resources.Resource.checkExists at Resource.java:341
2: org.eclipse.core.internal.resources.Resource.checkAccessible at Resource.java:215
3: org.eclipse.core.internal.resources.Resource.findMaxProblemSeverity at Resource.java:1051
4: org.eclipse.jdt.ui.ProblemsLabelDecorator.getPackageErrorTicksFromMarkers at ProblemsLabelDecorator.java:337
5: org.eclipse.jdt.ui.ProblemsLabelDecorator.computeAdornmentFlags at ProblemsLabelDecorator.java:212
6: org.eclipse.jdt.internal.ui.viewsupport.TreeHierarchyLayoutProblemsDecorator.computePackageAdornmentFlags at TreeHierarchyLayoutProblemsDecorator.java:47
7: org.eclipse.jdt.inter